# Sky source review

Interactive review of a sky position in OVRO-LWA time–frequency data. Enter a
**coordinate string** in the review UI — ICRS degrees (`RA, Dec`) or a source name —
then click **Apply** to load the heatmap. While typing a name (first character is a letter), a dropdown of matching entries from
`known_sources.yaml` appears; pick one or keep typing. Numeric-first input is RA/Dec.

For the active coordinate and **heatmap method**:

1. Build a time × frequency map (tracked pixel, patch statistic, patch maximum, or
   Gaussian patch fit — same options as `jupiter_flux_review.ipynb`, plus `mad`, `std`,
   `mean`, `min`).
2. **Click** a cell in the heatmap to load that slice in **astrowidget.SkyWidget**
   centered on the target (same sky-widget pattern as `jupiter_flux_review.ipynb`:
   `set_dataset`, patched per-time WCS, fixed `center=`).

Launch with: `pixi run jupyter lab`

**Run cells in order** (config → imports → helpers → class → optional Dask → launch UI).


In [ ]:
# Edit before running if your paths or cuts differ.
from pathlib import Path

## Checked again when the review UI starts; fix typos before launching.
ZARR_PATH = Path("/fast/claw/I-Clean-Snapshot-20250120-LST4-5.zarr/")

# Optional default for the Coordinate field (edit in the review UI after launch).
COORDINATE_STRING = ""

# Fall back to NED ObjectLookup when SkyCoord.from_name fails (requires network).
USE_NED_FALLBACK = True
NED_TIMEOUT_S = 10.0

# Known source names for autocomplete. Resolved relative to cwd and notebooks/.
KNOWN_SOURCES_PATH = Path("known_sources.yaml")

PATCH_SCALE = 5.0  # patch half-width = ceil(scale * max beam FWHM in pixels)
SKY_FOV_DEG = 8.0

# Default heatmap fill: tracked pixel, patch stats, patch_max, or patch_fit
HEATMAP_METHOD = "dynamic_spectrum"
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 10.0

# Zarr read pattern (matches jupiter_flux_review.ipynb)
ZARR_LM_CHUNK = 512

# Set False to scan the image centre for the first finite time index (extra I/O)
SKIP_FIRST_VALID_SKY_SCAN = True

# Optional distributed Dask (default off — Jupiter notebook uses none)
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 6
DASK_MEMORY_LIMIT = "16GiB"



In [ ]:
import warnings

warnings.filterwarnings("ignore")

import math
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import ovro_lwa_portal as ovro
from ovro_lwa_portal import resolve_coordinate_string
import panel as pn
import param
import xarray as xr
import astropy.units as u
from astropy.coordinates import SkyCoord
from bokeh.events import Tap
from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, LinearColorMapper
from bokeh.palettes import Inferno256, Magma256
from bokeh.plotting import figure
from astrowidget import SkyWidget

from ovro_lwa_portal.accessor import format_radec_sexagesimal
from ovro_lwa_portal.viz.pipeline_qa_app import (
    _capture_ipython_io_loop,
    ACTIVITY_LOG_HEIGHT_PX,
    ScrollLog,
    _format_activity_log_html,
    _patch_astrowidget_get_wcs,
    _push_panel_layout,
    _schedule_ipython_main,
    bind_sky_widget_dataset,
)

_patch_astrowidget_get_wcs()
pn.extension("bokeh", sizing_mode="stretch_width")
_capture_ipython_io_loop()

HEATMAP_METHOD_OPTIONS = [
    "dynamic_spectrum",
    "patch_max",
    "patch_fit",
    "mad",
    "std",
    "mean",
    "min",
]


In [ ]:
import re

import yaml


def filter_known_source_names(text: str, names: list[str]) -> list[str]:
    """``includes`` match on known names when input starts with a letter."""
    stripped = text.lstrip()
    if not stripped or not stripped[0].isalpha():
        return []
    query = stripped.casefold()
    return [name for name in names if query in name.casefold()]


def resolve_known_sources_path(path: Path) -> Path | None:
    """Find ``known_sources.yaml`` relative to cwd or ``notebooks/``."""
    for candidate in (path, Path.cwd() / path, Path.cwd() / "notebooks" / path.name):
        if candidate.is_file():
            return candidate.resolve()
    return None


def load_known_sources(path: Path) -> list[str]:
    """Load source name completion labels from a YAML file."""
    resolved = resolve_known_sources_path(path)
    if resolved is None:
        return []

    with resolved.open(encoding="utf-8") as fh:
        payload = yaml.safe_load(fh) or {}

    names: list[str] = []
    for entry in payload.get("sources") or []:
        if isinstance(entry, str):
            name = entry.strip()
        else:
            name = str(entry.get("name", "")).strip()
        if name:
            names.append(name)
    names.sort(key=str.casefold)
    return names


def build_source_from_coordinate(label: str, coord: SkyCoord) -> dict:
    """Build a source record from a resolved sky position."""
    gal = coord.galactic
    name = label if len(label) <= 48 else f"{label[:45]}…"
    return {
        "name": name,
        "l": float(gal.l.deg),
        "b": float(gal.b.deg),
        "ra": float(coord.ra.deg),
        "dec": float(coord.dec.deg),
    }


def lst_hours_for_dataset(ds: xr.Dataset) -> np.ndarray:
    """Mean local sidereal time (hours) for each dataset time sample."""
    from astropy.coordinates import EarthLocation
    from astropy.time import Time
    from astropy.utils.iers import conf as iers_conf

    observatory = EarthLocation.of_site("ovro")
    mjd = np.asarray(ds.coords["time"].values, dtype=np.float64)
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        times = Time(mjd, format="mjd", scale="utc")
        lst_deg = np.asarray(times.sidereal_time("mean", longitude=observatory.lon).deg)
    finally:
        iers_conf.auto_download = orig
    return np.mod(lst_deg / 15.0, 24.0)


def first_valid_sky_slice(dataset: xr.Dataset, freq_idx: int | None = None) -> tuple[int, int]:
    """First time index with finite SKY at the image centre."""
    fi = dataset.sizes["frequency"] // 2 if freq_idx is None else int(freq_idx)
    center = dataset.sizes["l"] // 2
    ts = dataset["SKY"].isel(polarization=0, frequency=fi, l=center, m=center)
    data = ts.data
    vals = np.asarray(data.compute() if hasattr(data, "compute") else data)
    valid = np.flatnonzero(np.isfinite(vals))
    if valid.size == 0:
        raise ValueError("No finite SKY data found at the image center.")
    return int(valid[0]), fi


@dataclass
class HeatmapLoad:
    """Values and optional accessor results for one source/method pair."""

    values: np.ndarray
    patch_fit_result: object | None = None
    patch_stat_result: object | None = None


_PROGRESS_STAGE_LABELS = {
    "track": "Pixel track",
    "extract": "Pixel I/O",
    "reduce": "Statistics",
    "fit": "Patch fit",
}


def compute_source_heatmap(
    dataset: xr.Dataset,
    src: dict,
    *,
    method: str,
    scale: float,
    patch_fit_max_reduced_chi_squared: float,
    progress_callback: Callable[[str, int, int, str], None] | None = None,
) -> HeatmapLoad:
    """Build the (time, frequency) array used to fill the heatmap."""
    ra = float(src["ra"])
    dec = float(src["dec"])
    if method == "dynamic_spectrum":
        da = dataset.radport.dynamic_spectrum(
            ra=ra, dec=dec, progress_callback=progress_callback
        )
        return HeatmapLoad(np.asarray(da.values, dtype=np.float64))
    if method == "patch_fit":
        fit = dataset.radport.patch_fit(
            ra=ra,
            dec=dec,
            scale=scale,
            max_reduced_chi_squared=patch_fit_max_reduced_chi_squared,
            allow_position_offset=True,
            progress_callback=progress_callback,
        )
        return HeatmapLoad(np.asarray(fit.peak_map.values, dtype=np.float64), patch_fit_result=fit)
    if method == "patch_max":
        result = dataset.radport.patch_statistic(
            ra=ra,
            dec=dec,
            statistic="max",
            scale=scale,
            progress_callback=progress_callback,
        )
        return HeatmapLoad(np.asarray(result.stat_map.values, dtype=np.float64), patch_stat_result=result)
    if method in ("mad", "std", "mean", "min"):
        result = dataset.radport.patch_statistic(
            ra=ra,
            dec=dec,
            statistic=method,
            scale=scale,
            progress_callback=progress_callback,
        )
        return HeatmapLoad(np.asarray(result.stat_map.values, dtype=np.float64), patch_stat_result=result)
    msg = f"Unknown heatmap method {method!r}; expected one of {HEATMAP_METHOD_OPTIONS}"
    raise ValueError(msg)


def diagnose_heatmap_coverage(
    dataset: xr.Dataset,
    src: dict,
    values: np.ndarray,
    *,
    method: str,
    patch_fit_max_reduced_chi_squared: float,
) -> str:
    """Explain missing (NaN) heatmap cells for logging in the review UI."""
    ra = float(src["ra"])
    dec = float(src["dec"])
    n_times, n_freqs = values.shape
    finite = np.isfinite(values)
    n_finite = int(finite.sum())
    if n_finite == finite.size:
        return ""

    lines: list[str] = []
    try:
        _li, _mi, visible = dataset.radport._compute_pixel_track(ra, dec)
        n_visible = int(np.sum(visible))
        lines.append(
            f"Sky footprint: source visible on {n_visible}/{n_times} time steps "
            f"in this dataset (gray = not in FOV or no finite data at the tracked pixel/patch)."
        )
        if n_visible == 0:
            lines.append(
                "The target position never falls inside this snapshot's image — "
                "common when the Zarr pointing differs from the requested sky position."
            )
        elif n_visible < n_times:
            lines.append(
                f"{n_times - n_visible} time step(s) are off-field; those rows stay gray."
            )
    except Exception as exc:
        lines.append(f"Could not evaluate sky footprint: {exc}")

    if n_finite and n_finite < finite.size:
        lines.append(
            f"Finite heatmap cells: {n_finite}/{finite.size} "
            f"({100.0 * n_finite / finite.size:.1f}%)."
        )
    elif n_finite == 0:
        lines.append("No finite heatmap cells.")

    if method == "patch_fit":
        lines.append(
            "Patch-fit also masks cells where reduced χ² exceeds "
            f"{patch_fit_max_reduced_chi_squared:g}."
        )
    return " ".join(lines)


def _heatmap_index_from_coord(coord: float, n: int) -> int:
    if n <= 0:
        return 0
    return int(np.clip(int(np.floor(float(coord))), 0, n - 1))


def _format_lst_hour_label(lst_hour: float) -> str:
    hour = int(round(float(lst_hour))) % 24
    return f"{hour:02d}h"


def _format_scalar_hover(value: float, *, fmt: str = ".3g") -> str:
    if np.isfinite(value):
        return format(float(value), fmt)
    return "n/a"


def _color_mapper(values: np.ndarray, *, palette=Magma256) -> LinearColorMapper:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return LinearColorMapper(palette=palette, low=0.0, high=1.0, nan_color="#9e9e9e")
    lo, hi = np.percentile(finite, [2, 98])
    if hi <= lo:
        hi = lo + 1.0
    return LinearColorMapper(
        palette=palette,
        low=float(lo),
        high=float(hi),
        nan_color="#9e9e9e",
    )


def _row_hover(arr: np.ndarray) -> list[str]:
    return [_format_scalar_hover(float(v)) for v in arr.ravel()]


def _patch_fit_hover_columns(fit: object) -> dict[str, list[str]]:
    """Pre-formatted Bokeh hover fields for patch-fit (same as Jupiter notebook)."""
    chi2 = np.asarray(fit.reduced_chi_squared_map.values, dtype=np.float64)
    peak = np.asarray(fit.peak_map.values, dtype=np.float64)
    x_off = np.asarray(fit.x_offset_map.values, dtype=np.float64)
    y_off = np.asarray(fit.y_offset_map.values, dtype=np.float64)
    pmax = np.asarray(fit.patch_max_map.values, dtype=np.float64)
    accepted = np.asarray(fit.fit_accepted_map.values, dtype=bool)
    peak_ra, peak_dec = fit.peak_radec_maps()
    ra = np.asarray(peak_ra.values, dtype=np.float64)
    dec = np.asarray(peak_dec.values, dtype=np.float64)

    peak_ra_display: list[str] = []
    peak_dec_display: list[str] = []
    for r, d in zip(ra.ravel(), dec.ravel(), strict=True):
        ra_s, dec_s = format_radec_sexagesimal(float(r), float(d))
        peak_ra_display.append(ra_s)
        peak_dec_display.append(dec_s)

    return {
        "chi2_display": _row_hover(chi2),
        "peak_ra_display": peak_ra_display,
        "peak_dec_display": peak_dec_display,
        "offset_display": [
            (
                f"({x:.2f}, {y:.2f})"
                if np.isfinite(x) and np.isfinite(y)
                else "n/a"
            )
            for x, y in zip(x_off.ravel(), y_off.ravel(), strict=True)
        ],
        "fit_accepted_display": ["yes" if a else "no" for a in accepted.ravel()],
        "patch_max_display": _row_hover(pmax),
        "peak_flux_display": [
            f"{v:.3g} (masked)" if not np.isfinite(v) else f"{v:.3g}"
            for v in peak.ravel()
        ],
    }


def _format_patch_fit_diagnostics(fit: object, time_idx: int, freq_idx: int) -> str:
    diag = fit.cell_diagnostics(time_idx=time_idx, frequency_idx=freq_idx)
    accepted = "yes" if diag["fit_accepted"] else "no (χ² above cut)"
    peak = diag["peak"]
    peak_s = f"{peak:.3g}" if np.isfinite(peak) else "n/a (masked)"
    return (
        f"**patch_fit** t={time_idx} f={freq_idx}: accepted={accepted}, "
        f"χ²_red={diag['reduced_chi_squared']:.3g}, peak={peak_s} Jy, "
        f"peak RA/Dec=({diag['peak_ra']}, {diag['peak_dec']}), "
        f"offset=({diag['x_offset_pixels']:.2f}, {diag['y_offset_pixels']:.2f}) px, "
        f"patch_max={diag['patch_max']:.3g} Jy"
    )




In [ ]:
class SourceReview(param.Parameterized):
    """Sky-position heatmap + SkyWidget review (Jupiter-style Panel UI)."""

    coordinate_string = param.String(
        default="",
        doc="ICRS RA/Dec in degrees or a source name (Apply to load).",
    )
    heatmap_method = param.Selector(
        default="mad",
        objects=HEATMAP_METHOD_OPTIONS,
        doc="Quantity plotted in the time–frequency heatmap.",
    )
    loading = param.Boolean(default=False)
    status = param.String(default="Opening Zarr store…")
    log_text = param.String(default="")

    def __init__(
        self,
        zarr_path: Path,
        *,
        coordinate_string: str = "",
        known_sources_path: Path | None = None,
        patch_scale: float,
        sky_fov_deg: float,
        patch_fit_max_reduced_chi_squared: float,
        **params,
    ) -> None:
        self._zarr_path = Path(zarr_path)
        self._known_sources_path = Path(known_sources_path) if known_sources_path else None
        if self._known_sources_path is not None:
            self._known_source_names = load_known_sources(self._known_sources_path)
        else:
            self._known_source_names = []
        self._patch_scale = float(patch_scale)
        self._sky_fov_deg = float(sky_fov_deg)
        self._patch_fit_max_chi2 = float(patch_fit_max_reduced_chi_squared)
        self._scroll_log = ScrollLog()
        self._dataset: xr.Dataset | None = None
        self._cache: dict[tuple[str, str], HeatmapLoad] = {}
        self._heatmap_values: np.ndarray | None = None
        self._patch_fit_result: object | None = None
        self._patch_stat_result: object | None = None
        self._current_source: dict | None = None
        self._coord: SkyCoord | None = None
        self._lst_hours: np.ndarray | None = None
        self._freq_mhz: np.ndarray | None = None
        self._sky_widget: SkyWidget | None = None
        self._time_idx = 0
        self._freq_idx = 0
        self._default_time_idx = 0
        self._default_freq_idx = 0
        self._heatmap_job_id = 0

        super().__init__(coordinate_string=coordinate_string.strip(), **params)

        self._heatmap_pane = pn.pane.Bokeh(height=420, sizing_mode="stretch_width")
        self._sky_container = widgets.VBox(
            children=[widgets.HTML("<i>Sky view loads after the Zarr store opens.</i>")],
            layout=widgets.Layout(width="100%", min_height="620px"),
        )
        self._sky_pane = pn.pane.IPyWidget(self._sky_container, height=620, sizing_mode="stretch_width")
        self._status_pane = pn.pane.Markdown("")
        self._log_pane = pn.pane.HTML(
            _format_activity_log_html(""),
            sizing_mode="stretch_width",
            height=ACTIVITY_LOG_HEIGHT_PX,
        )
        self._method_selector = pn.widgets.Select.from_param(
            self.param.heatmap_method,
            name="Heatmap method",
            width=220,
        )
        self._spinner = pn.indicators.LoadingSpinner(value=False, size=24, name="")
        initial_coord = coordinate_string.strip()
        self._coord_input = pn.widgets.AutocompleteInput(
            name="Coordinate",
            value=initial_coord,
            value_input=initial_coord,
            placeholder="RA°, Dec° or source name — Enter or Apply to load",
            options=filter_known_source_names(initial_coord, self._known_source_names),
            search_strategy="includes",
            case_sensitive=False,
            restrict=False,
            min_characters=1,
            sizing_mode="stretch_width",
        )
        self._coord_input.param.watch(self._on_coord_value_input, "value_input")
        self._coord_input.param.watch(self._on_coord_value, "value")
        self._coord_apply = pn.widgets.Button(name="Apply", button_type="primary", width=90)
        self._coord_apply.on_click(self._on_apply_coordinate)
        self._layout = pn.Column(
            pn.Row(
                self._coord_input,
                self._coord_apply,
                self._method_selector,
                self._spinner,
                sizing_mode="stretch_width",
                margin=(0, 0, 8, 0),
            ),
            pn.Column(
                pn.pane.Markdown("**Activity log**"),
                self._log_pane,
                sizing_mode="stretch_width",
            ),
            self._status_pane,
            self._heatmap_pane,
            self._sky_pane,
            sizing_mode="stretch_width",
            max_width=1048,
        )
        self.param.watch(self._on_heatmap_method_change, "heatmap_method")
        if self._known_sources_path is not None:
            resolved_sources = resolve_known_sources_path(self._known_sources_path)
            if resolved_sources is None:
                self._log(
                    f"WARNING: Known sources file not found: {self._known_sources_path} "
                    f"(cwd={Path.cwd()})"
                )
            elif self._known_source_names:
                self._log(
                    f"Known sources: {len(self._known_source_names)} names from {resolved_sources}"
                )
        self._log(f"Zarr: {self._zarr_path}")
        if self.coordinate_string.strip():
            self._apply_coordinate_from_field()
        else:
            self._set_status(
                "**Enter a coordinate** (`RA°, Dec°` or source name) and click **Apply**."
            )
        try:
            resolved = ovro.validate_local_zarr_store(self._zarr_path)
            if resolved != self._zarr_path.resolve():
                self._log(f"Resolved Zarr path: {resolved}")
                self._zarr_path = resolved
        except (FileNotFoundError, DataSourceError) as exc:
            self.loading = False
            self._spinner.value = False
            self._log(f"ERROR: {exc}")
            self._set_status(
                "**Invalid Zarr path** — correct `ZARR_PATH` in the config cell "
                "and re-run the launch cell."
            )
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return
        self._log_dask_dashboard()
        self._open_dataset()

    @property
    def panel(self) -> pn.Column:
        return self._layout

    def _active_coordinate_text(self) -> str:
        typing = (self._coord_input.value_input or "").strip()
        committed = (self._coord_input.value or "").strip()
        if not typing and not committed:
            return (self.coordinate_string or "").strip()
        if committed and typing and committed.casefold() != typing.casefold():
            # AutocompleteInput can leave value_input as a typed prefix after a
            # dropdown pick; value holds the committed completion.
            if committed.casefold().startswith(typing.casefold()) and (
                committed in self._known_source_names or len(committed) > len(typing)
            ):
                return committed
            return typing
        return typing or committed or (self.coordinate_string or "").strip()

    def _coord_match_options(self, text: str) -> list[str]:
        return filter_known_source_names(text, self._known_source_names)

    def _on_coord_value_input(self, event) -> None:
        text = event.new or ""
        self.coordinate_string = text
        options = self._coord_match_options(text)
        if list(self._coord_input.options) != options:
            self._coord_input.options = options

    def _on_coord_value(self, event) -> None:
        text = str(event.new or "").strip()
        if not text:
            return
        self.coordinate_string = text
        with param.parameterized.discard_events(self._coord_input):
            self._coord_input.value_input = text
        self._on_apply_coordinate()

    def _apply_coordinate_from_field(self) -> bool:
        coord_text = self._active_coordinate_text().strip()
        if not coord_text:
            self._log("WARNING: Coordinate is empty — enter RA/Dec or a source name.")
            return False

        resolution, messages = resolve_coordinate_string(
            coord_text,
            use_ned_fallback=USE_NED_FALLBACK,
            ned_timeout=NED_TIMEOUT_S,
            known_source_names=frozenset(
                name.casefold() for name in self._known_source_names
            ),
        )
        for message in messages:
            self._log(message)
        if resolution is None:
            self._log(
                f"WARNING: Could not resolve coordinate {coord_text!r} "
                "(expected RA/Dec in degrees or a resolvable name)."
            )
            return False

        coord = resolution.coord
        label = resolution.canonical_name or coord_text
        self._current_source = build_source_from_coordinate(label, coord)
        self._cache.clear()
        src = self._current_source
        resolver_note = f" [{resolution.resolver}]"
        self._log(
            f"Coordinate {coord_text!r} → RA={src['ra']:.4f}°, Dec={src['dec']:.4f}°{resolver_note}."
        )
        return True

    def _on_apply_coordinate(self, _event: object | None = None) -> None:
        resolved = self._active_coordinate_text()
        if resolved:
            self.coordinate_string = resolved
            with param.parameterized.discard_events(self._coord_input):
                self._coord_input.value = resolved
                self._coord_input.value_input = resolved
        if not self._apply_coordinate_from_field():
            return
        if self._dataset is None:
            return
        self._load_heatmap()

    def _heatmap_method_label(self) -> str:
        labels = {
            "dynamic_spectrum": "tracked centre pixel",
            "patch_max": "patch maximum",
            "patch_fit": "Gaussian patch fit (peak)",
            "mad": "patch MAD",
            "std": "patch std",
            "mean": "patch mean",
            "min": "patch min",
        }
        return labels.get(self.heatmap_method, self.heatmap_method)

    def _set_status(self, text: str) -> None:
        self.status = text
        self._status_pane.object = text

    @param.depends("log_text", watch=True)
    def _sync_log_pane(self) -> None:
        self._log_pane.object = _format_activity_log_html(self.log_text)

    def _sync_log(self) -> None:
        self.log_text = self._scroll_log.text

    def _log(self, message: str) -> None:
        self._scroll_log.append(message)
        self._sync_log()
        _push_panel_layout(self._layout, self._log_pane)

    def _log_dask_dashboard(self) -> None:
        try:
            from dask.distributed import get_client

            client = get_client()
            self._log(f"Dask dashboard: {client.dashboard_link}")
        except Exception:
            pass

    def _heatmap_progress_callback(self) -> Callable[[str, int, int, str], None]:
        last_key: dict[str, tuple[int, str]] = {}

        def _callback(stage: str, current: int, total: int, message: str) -> None:
            if stage in ("extract", "track") and total > 1:
                key = (current, message)
                if current not in (0, total) and last_key.get(stage) == key:
                    return
                last_key[stage] = key
            label = _PROGRESS_STAGE_LABELS.get(stage, stage)
            if "in progress" in message or (stage == "extract" and current < total):
                text = f"{label}: {message}"
            else:
                pct = int(round(100.0 * int(current) / int(total))) if total else 0
                text = f"{label}: {message} ({current}/{total}, {pct}%)"

            def _push() -> None:
                self._log(text)

            _schedule_ipython_main(_push)

        return _callback


    def _open_dataset(self) -> None:
        self.loading = True
        self._spinner.value = True
        _push_panel_layout(self._layout, self._spinner, self._log_pane)

        def _work() -> None:
            try:
                t_open = time.perf_counter()

                def _log_elapsed(msg: str, t0: float) -> None:
                    elapsed = time.perf_counter() - t0
                    self._log(f"{msg} ({elapsed:.1f} s)")

                _schedule_ipython_main(
                    lambda: self._log(
                        f"Opening {self._zarr_path} (chunks='auto', l/m={ZARR_LM_CHUNK})…"
                    )
                )
                t_meta = time.perf_counter()
                ds = ovro.open_dataset(self._zarr_path, chunks="auto").chunk(
                    {"l": ZARR_LM_CHUNK, "m": ZARR_LM_CHUNK}
                )
                _schedule_ipython_main(
                    lambda: _log_elapsed("Zarr opened", t_meta)
                )

                if SKIP_FIRST_VALID_SKY_SCAN:
                    t0, f0 = 0, int(ds.sizes["frequency"]) // 2
                    _schedule_ipython_main(
                        lambda: self._log(
                            f"Using default slice time={t0}, freq={f0} "
                            "(SKIP_FIRST_VALID_SKY_SCAN=True)"
                        )
                    )
                else:
                    _schedule_ipython_main(
                        lambda: self._log(
                            "Scanning centre pixel for first valid time index…"
                        )
                    )
                    t_scan = time.perf_counter()
                    t0, f0 = first_valid_sky_slice(ds)
                    _schedule_ipython_main(
                        lambda: _log_elapsed("Centre-pixel scan complete", t_scan)
                    )

                _schedule_ipython_main(
                    lambda: self._log("Computing LST labels for time axis…")
                )
                t_lst = time.perf_counter()
                lst_hours = lst_hours_for_dataset(ds)
                freq_mhz = np.asarray(ds.coords["frequency"].values, dtype=np.float64) / 1e6
                _schedule_ipython_main(
                    lambda: _log_elapsed(
                        f"Coordinates ready — {int(ds.sizes['time'])}×{int(ds.sizes['frequency'])} "
                        f"heatmap grid; total open",
                        t_lst,
                    )
                )
                _schedule_ipython_main(
                    lambda: self._log(
                        f"Open pipeline finished in {time.perf_counter() - t_open:.1f} s"
                    )
                )
            except Exception as exc:
                _schedule_ipython_main(lambda: self._finish_open(None, None, None, None, None, exc))
                return
            _schedule_ipython_main(
                lambda: self._finish_open(ds, t0, f0, lst_hours, freq_mhz, None)
            )

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_open(
        self,
        ds: xr.Dataset | None,
        default_time_idx: int | None,
        default_freq_idx: int | None,
        lst_hours: np.ndarray | None,
        freq_mhz: np.ndarray | None,
        error: BaseException | None,
    ) -> None:
        if error is not None:
            self.loading = False
            self._spinner.value = False
            err_text = str(error).strip()
            self._log(f"ERROR: {err_text}")
            first_line = err_text.splitlines()[0] if err_text else repr(error)
            self._set_status(f"**Load failed:** {first_line}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert ds is not None and lst_hours is not None and freq_mhz is not None
        assert default_time_idx is not None and default_freq_idx is not None

        self._dataset = ds
        self._lst_hours = lst_hours
        self._freq_mhz = freq_mhz
        self._default_time_idx = int(default_time_idx)
        self._default_freq_idx = int(default_freq_idx)
        self._time_idx = self._default_time_idx
        self._freq_idx = self._default_freq_idx

        self._log(
            f"Opened — {int(ds.sizes['time'])} times × {int(ds.sizes['frequency'])} freqs, "
            f"{int(ds.sizes['l'])}×{int(ds.sizes['m'])} px, WCS={ds.radport.has_wcs}."
        )
        self._mount_sky_widget(ds)
        self.loading = False
        self._spinner.value = False
        if self._current_source is not None:
            self._load_heatmap()

    def _on_heatmap_method_change(self, *_events) -> None:
        if self._current_source is None or self._dataset is None or self.loading:
            return
        self._load_heatmap()

    def _load_heatmap(self) -> None:
        if self._dataset is None or self._current_source is None:
            return
        src = self._current_source
        method = str(self.heatmap_method)
        cache_key = (self.coordinate_string.strip(), method)
        if cache_key in self._cache:
            self._apply_heatmap(src, self._cache[cache_key])
            return

        self._heatmap_job_id += 1
        job_id = self._heatmap_job_id
        self.loading = True
        self._spinner.value = True
        n_times = int(self._dataset.sizes["time"])
        n_freqs = int(self._dataset.sizes["frequency"])
        label = src["name"]
        self._log(
            f"Computing {self._heatmap_method_label()} for {label} "
            f"({n_times} times × {n_freqs} freqs, "
            f"RA={src['ra']:.4f}°, Dec={src['dec']:.4f}°)…"
        )
        self._set_status(f"Computing **{self._heatmap_method_label()}** for **{label}**…")
        _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)

        def _work() -> None:
            t0 = time.perf_counter()
            try:
                payload = compute_source_heatmap(
                    self._dataset,
                    src,
                    method=method,
                    scale=self._patch_scale,
                    patch_fit_max_reduced_chi_squared=self._patch_fit_max_chi2,
                    progress_callback=self._heatmap_progress_callback(),
                )
            except Exception as exc:
                _schedule_ipython_main(
                    lambda: self._finish_heatmap(src, None, exc, job_id, t0)
                )
                return
            _schedule_ipython_main(
                lambda: self._finish_heatmap(src, payload, None, job_id, t0)
            )

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_heatmap(
        self,
        src: dict,
        payload: HeatmapLoad | None,
        error: BaseException | None,
        job_id: int,
        started_at: float,
    ) -> None:
        if job_id != self._heatmap_job_id:
            return
        elapsed_s = time.perf_counter() - started_at
        self.loading = False
        self._spinner.value = False
        if error is not None:
            self._log(f"ERROR ({src['name']}): {error}")
            self._set_status(f"**Heatmap failed for {src['name']}:** {error}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert payload is not None
        cache_key = (self.coordinate_string.strip(), str(self.heatmap_method))
        self._cache[cache_key] = payload
        arr = payload.values
        finite = arr[np.isfinite(arr)]
        if finite.size:
            self._log(
                f"{src['name']} ({self._heatmap_method_label()}): "
                f"range [{float(finite.min()):.3g}, {float(finite.max()):.3g}] "
                f"({finite.size}/{arr.size} finite cells)"
            )
        else:
            self._log(f"{src['name']} ({self._heatmap_method_label()}): no finite values")
        if self._dataset is not None:
            hint = diagnose_heatmap_coverage(
                self._dataset,
                src,
                arr,
                method=str(self.heatmap_method),
                patch_fit_max_reduced_chi_squared=self._patch_fit_max_chi2,
            )
            if hint:
                self._log(f"Hint: {hint}")
        self._log(
            f"Finished {src['name']} ({self._heatmap_method_label()}) in {elapsed_s:.1f} s"
        )
        self._apply_heatmap(src, payload)

    def _apply_heatmap(self, src: dict, payload: HeatmapLoad) -> None:
        self._current_source = src
        self._heatmap_values = payload.values
        self._patch_fit_result = payload.patch_fit_result
        self._patch_stat_result = payload.patch_stat_result
        self._coord = SkyCoord(ra=src["ra"] * u.deg, dec=src["dec"] * u.deg, frame="icrs")
        self._time_idx, self._freq_idx = self._default_slice(payload.values)

        ra_h = self._coord.ra.to_string(unit=u.hour, precision=1)
        dec_s = self._coord.dec.to_string(unit=u.deg, precision=1)
        self._set_status(
            f"**{src['name']}** — l={src['l']:.2f}°, b={src['b']:.2f}°, "
            f"RA={ra_h}, Dec={dec_s} · "
            f"Heatmap: **{self._heatmap_method_label()}** (scale={self._patch_scale:g}) · "
            "**Click the heatmap** to inspect a time/frequency slice."
        )
        self._heatmap_pane.object = self._build_heatmap_figure(payload.values)
        self._update_sky(self._time_idx, self._freq_idx)
        _push_panel_layout(
            self._layout, self._status_pane, self._heatmap_pane, self._sky_pane, self._log_pane
        )

    def _default_slice(self, values: np.ndarray) -> tuple[int, int]:
        finite = np.argwhere(np.isfinite(values))
        if finite.size:
            t_idx, f_idx = finite[len(finite) // 2]
            return int(t_idx), int(f_idx)
        return self._default_time_idx, self._default_freq_idx

    def _mount_sky_widget(self, ds: xr.Dataset) -> None:
        """Match ``jupiter_flux_review.ipynb`` (proven with per-time CRVAL)."""
        widget = SkyWidget()
        widget.colormap = "inferno"
        widget.background_survey = ""
        widget.invert_horizontal_pan = True
        max_size = max(256, int(ds.sizes["l"]) // 2)
        bind_sky_widget_dataset(widget, ds, max_size=max_size)
        self._sky_widget = widget
        self._sky_container.children = [widget]

    def _update_sky(self, time_idx: int, freq_idx: int) -> None:
        widget = self._sky_widget
        coord = self._coord
        if widget is None or coord is None:
            return
        widget.update_slice(
            time_idx=int(time_idx),
            freq_idx=int(freq_idx),
            center=coord,
            fov=self._sky_fov_deg * u.deg,
            percentile_low=2,
            percentile_high=98,
        )
        send_state = getattr(widget, "send_state", None)
        if callable(send_state):
            send_state()

    def _on_heatmap_tap(self, time_idx: int, freq_idx: int) -> None:
        self._time_idx = time_idx
        self._freq_idx = freq_idx
        src = self._current_source
        if src is None or self._lst_hours is None or self._freq_mhz is None:
            return

        lst = _format_lst_hour_label(float(self._lst_hours[time_idx]))
        freq = float(self._freq_mhz[freq_idx])
        val = (
            float(self._heatmap_values[time_idx, freq_idx])
            if self._heatmap_values is not None
            else float("nan")
        )
        val_s = f"{val:.3g}" if np.isfinite(val) else "n/a"

        coord = self._coord
        assert coord is not None
        ra_h = coord.ra.to_string(unit=u.hour, precision=1)
        dec_s = coord.dec.to_string(unit=u.deg, precision=1)
        track_note = ""
        if self._dataset is not None:
            try:
                li, mi = self._dataset.radport.coords_to_pixel(
                    float(coord.ra.deg), float(coord.dec.deg), time_idx=time_idx
                )
                tr_ra, tr_dec = self._dataset.radport.pixel_to_coords(li, mi, time_idx=time_idx)
                track_note = (
                    f" · tracked@slice RA={tr_ra:.4f}°, Dec={tr_dec:.4f}° "
                    f"(pix {li},{mi})"
                )
            except ValueError as exc:
                track_note = f" · tracked@slice: {exc}"
        status = (
            f"**{src['name']}** · LST {lst}, {freq:.1f} MHz (t={time_idx}, f={freq_idx}) · "
            f"{self._heatmap_method_label()}={val_s} · target RA={ra_h}, Dec={dec_s}"
            f"{track_note}"
        )
        if self.heatmap_method == "patch_fit" and self._patch_fit_result is not None:
            status = f"{status}\n\n{_format_patch_fit_diagnostics(self._patch_fit_result, time_idx, freq_idx)}"
        self._set_status(status)
        self._log(
            f"Sky slice — {src['name']}, {self._heatmap_method_label()}={val_s}, "
            f"t={time_idx}, f={freq_idx} ({freq:.1f} MHz)."
        )
        self._update_sky(time_idx, freq_idx)
        _push_panel_layout(self._layout, self._status_pane, self._sky_pane, self._log_pane)

    def _build_heatmap_figure(self, values: np.ndarray):
        n_times, n_freqs = values.shape
        mapper = _color_mapper(values.astype(np.float64, copy=False))
        src = self._current_source
        title_name = src["name"] if src is not None else "source"
        method_label = self._heatmap_method_label()
        plot = figure(
            width=1000,
            height=400,
            title=f"{title_name} — {method_label} (click a cell for sky view)",
            x_range=(0, n_times),
            y_range=(0, n_freqs),
            tools="pan,wheel_zoom,reset,tap",
            active_drag="pan",
            active_tap="tap",
        )
        plot.image(
            image=[values.T.astype(np.float64, copy=False)],
            x=0,
            y=0,
            dw=n_times,
            dh=n_freqs,
            color_mapper=mapper,
        )

        time_idx, freq_idx = np.meshgrid(
            np.arange(n_times, dtype=int),
            np.arange(n_freqs, dtype=int),
            indexing="ij",
        )
        flat_time = time_idx.ravel()
        flat_freq = freq_idx.ravel()
        hover_data: dict[str, object] = {
            "x": flat_time + 0.5,
            "y": flat_freq + 0.5,
            "time_idx": flat_time,
            "freq_idx": flat_freq,
            "lst_hour": [
                _format_lst_hour_label(float(h))
                for h in self._lst_hours[flat_time]  # type: ignore[index]
            ],
            "freq_mhz": self._freq_mhz[flat_freq],  # type: ignore[index]
            "value_display": _row_hover(values),
        }
        tooltips: list[tuple[str, str]] = [
            ("LST hour", "@lst_hour"),
            ("Freq (MHz)", "@freq_mhz{0.1}"),
            ("Time idx", "@time_idx"),
            ("Freq idx", "@freq_idx"),
            ("Value", "@value_display"),
        ]
        if self.heatmap_method == "patch_fit" and self._patch_fit_result is not None:
            hover_data.update(_patch_fit_hover_columns(self._patch_fit_result))
            tooltips.extend(
                [
                    ("Patch max (Jy)", "@patch_max_display"),
                    ("χ²_red", "@chi2_display"),
                    ("Fit accepted", "@fit_accepted_display"),
                    ("Peak RA", "@peak_ra_display"),
                    ("Peak Dec", "@peak_dec_display"),
                    ("Offset (l,m px)", "@offset_display"),
                ]
            )

        hover_src = ColumnDataSource(data=hover_data)
        hover_renderer = plot.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=hover_src,
            fill_alpha=0,
            line_alpha=0,
        )
        plot.add_tools(HoverTool(renderers=[hover_renderer], tooltips=tooltips))

        def _axis_ticks(
            n: int, axis_values: np.ndarray, fmt: Callable[[float], str]
        ) -> tuple[list[float], dict[float, str]]:
            step = 1 if n <= 24 else int(np.ceil(n / 24))
            indices = range(0, n, step)
            ticks = [i + 0.5 for i in indices]
            labels = {tick: fmt(float(axis_values[i])) for tick, i in zip(ticks, indices, strict=True)}
            return ticks, labels

        x_ticks, x_labels = _axis_ticks(
            n_times, self._lst_hours, _format_lst_hour_label  # type: ignore[arg-type]
        )
        y_ticks, y_labels = _axis_ticks(
            n_freqs, self._freq_mhz, lambda v: f"{float(v):.1f}"  # type: ignore[arg-type]
        )
        plot.xaxis.ticker = FixedTicker(ticks=x_ticks)
        plot.yaxis.ticker = FixedTicker(ticks=y_ticks)
        plot.xaxis.major_label_overrides = x_labels
        plot.yaxis.major_label_overrides = y_labels
        plot.xaxis.axis_label = "LST hour"
        plot.yaxis.axis_label = "Frequency (MHz)"
        plot.xaxis.major_label_orientation = math.pi / 4

        def _on_tap(event: Tap) -> None:
            if event.x is None or event.y is None:
                return
            t_idx = _heatmap_index_from_coord(event.x, n_times)
            f_idx = _heatmap_index_from_coord(event.y, n_freqs)
            _schedule_ipython_main(lambda: self._on_heatmap_tap(t_idx, f_idx))

        plot.on_event(Tap, _on_tap)
        return plot


In [ ]:
# Optional — default matches jupiter_flux_review (no distributed Client).
if USE_DASK_CLIENT:
    from dask.distributed import Client, get_client

    try:
        dask_client = get_client()
    except ValueError:
        dask_client = Client(
            n_workers=DASK_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=False,
            memory_limit=DASK_MEMORY_LIMIT,
        )
    print(dask_client)
    print(f"Dashboard: {dask_client.dashboard_link}")
else:
    print("Dask Client disabled (same default as jupiter_flux_review).")


In [ ]:
ovro.validate_local_zarr_store(ZARR_PATH)

review = SourceReview(
    ZARR_PATH,
    coordinate_string=COORDINATE_STRING,
    known_sources_path=KNOWN_SOURCES_PATH,
    patch_scale=PATCH_SCALE,
    sky_fov_deg=SKY_FOV_DEG,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    heatmap_method=HEATMAP_METHOD,
)
review.panel


## Notes

- **Coordinate** — enter `"RA_deg, Dec_deg"` or a source name. When the first
  character is a letter, an `includes` dropdown lists hits from `known_sources.yaml` (pick one or
  keep typing; nothing is auto-filled). Press **Enter** or **Apply** to load. Numeric-first input
  skips matching (RA/Dec mode). Names resolve
  via `SkyCoord.from_name`, then NED ObjectLookup when enabled. Maps are cached
  per coordinate and heatmap method. Parse failures log a
  **WARNING** in the activity log.
- **Gray heatmap cells** — NaN when the target is **outside this Zarr snapshot's field of view** at
  that time, when SKY is masked at the tracked pixel/patch, or when patch_fit is rejected by the χ² cut.
  The activity log prints a footprint hint when many cells are missing.
- **Patch size** — `PATCH_SCALE` × max beam FWHM at each time step.
- **Zarr open** — `open_dataset(..., chunks="auto").chunk({"l": 512, "m": 512})` like `jupiter_flux_review.ipynb` (`ZARR_LM_CHUNK`). Set `SKIP_FIRST_VALID_SKY_SCAN=False` only if you need a centre-pixel time scan on open.
- **Progress** — dynamic spectrum reports **Pixel track** then **Pixel I/O** (per-time reads); patch methods report in the activity log (`Patch I/O`, then `Statistics` or `Patch fit`) with time-step counts.
- First extraction on a full cube can take tens of seconds per coordinate depending on Zarr I/O and time-axis length.
- **Dask (optional)** — leave `USE_DASK_CLIENT = False` unless you need the dashboard for debugging; point extractions use a local `threads` scheduler when a `Client` is active. If enabled and patch methods warn on memory, lower `DASK_WORKERS` or raise `DASK_MEMORY_LIMIT`.
